# Voice Recognition: Fine-tuning + Inference Testing

Fine-tunes the voice embedding model used by `models/voice/inference.py` (SpeechBrain ECAPA-TDNN + ArcFace - see `docs/VOICE_MODEL.md`), evaluates it, and runs an audio-based testing section (genuine vs. impostor pairs).

**Dataset:** [Voxceleb1 audio wav files for India celebrity](https://www.kaggle.com/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity) - downloaded here via `kagglehub` (a free Kaggle account + API token, stored as Colab Secrets, is required - never hard-coded in this notebook). No Google Drive mount is used; everything is saved to the local Colab runtime disk.

## 1. Setup

In [ ]:
import os, sys
REPO_URL = "https://github.com/Malik8122/Cancelable-Multimodal-Biometric-Authentication-for-Critical-Infrastructure.git"
REPO_DIR = "/content/repo"
REPO_BRANCH = "phase-1-foundation"

if not os.path.isdir(REPO_DIR):
    !git clone -q -b $REPO_BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q h5py speechbrain torchaudio kagglehub

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

## 2. Dataset

Uses `kagglehub` with a Kaggle API token stored as a **Colab Secret** (`Secrets` panel in the left sidebar - add `KAGGLE_USERNAME` and `KAGGLE_KEY`, never paste them directly into a cell). No Google Drive mount needed - `kagglehub` downloads straight to local Colab disk.

In [ ]:
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
dataset_path = kagglehub.dataset_download("gaurav41/voxceleb1-audio-wav-files-for-india-celebrity")
DATASET_ROOT = os.path.join(dataset_path, "vox1_indian", "content", "vox_indian")
assert os.path.isdir(DATASET_ROOT), f"Expected VoxCeleb1 speaker directories at {DATASET_ROOT}"
print("Dataset ready at:", DATASET_ROOT)

## 3. Build train/val/test datasets

Same `VoxCelebDataset` (`models/voice/dataset.py`) the Kaggle Kernel training run uses - per-identity 70/15/15 split, documented in `docs/VOICE_MODEL.md`.

In [ ]:
from models.voice.config import VoiceConfig
from models.voice.dataset import VoxCelebDataset

config = VoiceConfig()
train_dataset = VoxCelebDataset(DATASET_ROOT, mode="train", config=config)
val_dataset = VoxCelebDataset(DATASET_ROOT, mode="val", config=config)
test_dataset = VoxCelebDataset(DATASET_ROOT, mode="test", config=config)

print(f"Speakers: {train_dataset.num_speakers}")
print(f"Train/Val/Test utterances: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")

## 4. Visualize a sample

A quick sanity check of the preprocessing pipeline (`preprocessing/voice.py`) before spending GPU time training on it.

In [ ]:
import matplotlib.pyplot as plt

sample_mel, sample_label = train_dataset[0]
plt.figure(figsize=(8, 3))
plt.imshow(sample_mel.numpy(), aspect="auto", origin="lower")
plt.title(f"Log-mel filterbank (speaker label {sample_label})")
plt.xlabel("Frame")
plt.ylabel("Mel bin")
plt.colorbar()
plt.show()

## 5. Train

In [ ]:
from models.voice.train import train

OUTPUT_DIR = f"{REPO_DIR}/models/voice/saved"
CHECKPOINT_PATH = train(DATASET_ROOT, OUTPUT_DIR, config=config)
print("Saved checkpoint:", CHECKPOINT_PATH)

## 6. Evaluate on the held-out test set

In [ ]:
from models.voice.inference import VoiceEmbedder
from evaluation.voice_metrics import run_voice_experiment

fine_tuned_embedder = VoiceEmbedder(checkpoint_path=CHECKPOINT_PATH, device=DEVICE)
assert not fine_tuned_embedder.mock_mode, "Checkpoint failed to load - check the path above."

test_embeddings, test_labels = [], []
for index in range(len(test_dataset)):
    mel_tensor, label = test_dataset[index]
    pseudo_rgb = mel_tensor.unsqueeze(-1).repeat(1, 1, 3).numpy()
    test_embeddings.append(fine_tuned_embedder.extract_embedding(pseudo_rgb))
    test_labels.append(str(label))

report = run_voice_experiment(test_embeddings, test_labels)
print(f"EER: {report['eer']:.4f} | Accuracy: {report['accuracy']:.4f} | AUC: {report['auc']:.4f}")

## 7. Test on audio - genuine vs. impostor pair

Plays (if running interactively) and reports the cosine similarity for one same-speaker pair and one different-speaker pair from the test set.

In [ ]:
import numpy as np
from evaluation.metrics import cosine_similarity

rng = np.random.default_rng(42)
test_label_names = np.array(test_labels)

def pick_pair(same_speaker: bool):
    for _ in range(200):
        i, j = rng.choice(len(test_embeddings), size=2, replace=False)
        if (test_label_names[i] == test_label_names[j]) == same_speaker:
            return i, j
    raise RuntimeError("Could not find a suitable pair - try a larger test split.")

genuine_i, genuine_j = pick_pair(same_speaker=True)
impostor_i, impostor_j = pick_pair(same_speaker=False)
print("Genuine pair similarity:  ", cosine_similarity(test_embeddings[genuine_i], test_embeddings[genuine_j]))
print("Impostor pair similarity:", cosine_similarity(test_embeddings[impostor_i], test_embeddings[impostor_j]))

from IPython.display import Audio, display
genuine_wav_path, _ = test_dataset._items[genuine_i]
print("Sample genuine-pair audio (first utterance):")
display(Audio(str(genuine_wav_path)))

## Output

Checkpoints are saved locally under `models/voice/saved/` (`voice_embedder.pt`, `voice_embedder.h5`, `training_config.json`) - no Drive mount needed. Download them from the Colab file browser, or `git add`/commit them directly from a Colab terminal cell if you've configured git credentials, then push so they land in this repo the same way the Kaggle Kernel's output does.